# Project 1: Data Cleaning & Preparation

**Goal:** Clean the raw order dataset by handling missing values, duplicates, and incorrect formats, following the four-phase process:

1. Strategic Imputation
2. Integrity Audit (duplicates)
3. Standardization (dates, casing, whitespace, numeric precision)
4. Verification Gate (0% errors on unique IDs and date formats)

Every change made is logged in a **change log** table at the end, which is exported separately as a PDF for easeiness.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

SOURCE_FILE = "Dataset_for_Data_Analytics.xlsx"
OUTPUT_FILE = "Dataset_for_Data_Analytics_CLEANED.xlsx"

df = pd.read_excel(SOURCE_FILE)
change_log = []  # list of dicts: Change ID, Description, Impact, Status

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
df.head()

Loaded 1200 rows, 14 columns


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


## Step 0 — Initial Inspection
Understand shape, dtypes, and get a first read on data quality before touching anything.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   str           
 1   Date             1200 non-null   datetime64[us]
 2   CustomerID       1200 non-null   str           
 3   Product          1200 non-null   str           
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   str           
 7   PaymentMethod    1200 non-null   str           
 8   OrderStatus      1200 non-null   str           
 9   TrackingNumber   1200 non-null   str           
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       891 non-null    str           
 12  ReferralSource   1200 non-null   str           
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[us](1), float64(2), int64(2), str(9)


In [3]:
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity             0
UnitPrice            0
ShippingAddress      0
PaymentMethod        0
OrderStatus          0
TrackingNumber       0
ItemsInCart          0
CouponCode         309
ReferralSource       0
TotalPrice           0
dtype: int64


In [4]:
print("Duplicate OrderIDs:", df['OrderID'].duplicated().sum())
print("Duplicate TrackingNumbers:", df['TrackingNumber'].duplicated().sum())
print("Fully duplicated rows:", df.duplicated().sum())

Duplicate OrderIDs: 0
Duplicate TrackingNumbers: 0
Fully duplicated rows: 0


## Phase 1 — Strategic Imputation
*Handle the gaps. Don't just delete.*

`CouponCode` is the only column with missing values (no coupon applied at checkout — this is a legitimate business state, not corrupt data). Rather than dropping these rows (which would destroy real order records) or imputing a statistical mean/mode on a categorical field, we impute with an explicit `"No Coupon"` label so the column stays fully populated and analysis-ready.

In [5]:
missing_before = df['CouponCode'].isnull().sum()

df['CouponCode'] = df['CouponCode'].fillna('No Coupon')

change_log.append({
    'Change ID': 'CR001',
    'Description': f"Imputed {missing_before} missing 'CouponCode' values with label 'No Coupon'",
    'Impact': f"Preserved {missing_before} records; 0 rows dropped",
    'Status': 'Resolved'
})

print(f"Imputed {missing_before} missing CouponCode values.")
df['CouponCode'].value_counts()

Imputed 309 missing CouponCode values.


CouponCode
FREESHIP     313
No Coupon    309
WINTER15     292
SAVE10       286
Name: count, dtype: int64

## Phase 2 — Integrity Audit
*One truth, one record.* Check `OrderID` and `TrackingNumber` (the dataset's unique identifiers) for duplicates, and check for fully duplicated rows.

In [6]:
dupe_order_ids = df['OrderID'].duplicated().sum()
dupe_tracking = df['TrackingNumber'].duplicated().sum()
dupe_rows = df.duplicated().sum()

rows_before = len(df)
df = df.drop_duplicates()
rows_after = len(df)
rows_removed = rows_before - rows_after

change_log.append({
    'Change ID': 'CR002',
    'Description': f"Audited unique identifiers (OrderID, TrackingNumber) and full-row duplicates",
    'Impact': f"Found {dupe_order_ids} duplicate OrderIDs, {dupe_tracking} duplicate TrackingNumbers, "
              f"{dupe_rows} duplicate rows. Removed {rows_removed} duplicate row(s).",
    'Status': 'Resolved'
})

print(f"Duplicate OrderIDs: {dupe_order_ids}")
print(f"Duplicate TrackingNumbers: {dupe_tracking}")
print(f"Duplicate rows removed: {rows_removed}")
print(f"Rows remaining: {rows_after}")

Duplicate OrderIDs: 0
Duplicate TrackingNumbers: 0
Duplicate rows removed: 0
Rows remaining: 1200


## Phase 3 — Speak One Language
*Standardize formats so every record reads the same way.*

- Dates → ISO 8601 (`YYYY-MM-DD`)
- Text columns → trimmed whitespace + consistent casing
- Monetary columns → 2 decimal precision
- Recompute/validate `TotalPrice = Quantity × UnitPrice`

In [7]:
# --- Dates to ISO 8601 ---
df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')

change_log.append({
    'Change ID': 'CR003',
    'Description': "Standardized 'Date' column to ISO 8601 format (YYYY-MM-DD)",
    'Impact': f"{len(df)} records reformatted",
    'Status': 'Resolved'
})
df[['Date']].head()

,Date
0,2023-01-04
1,2024-08-23
2,2024-02-27
3,2023-10-15
4,2025-05-08


In [8]:
# --- Trim whitespace + consistent casing on text columns ---
text_cols = ['Product', 'ShippingAddress', 'PaymentMethod', 'OrderStatus',
             'CouponCode', 'ReferralSource', 'OrderID', 'CustomerID', 'TrackingNumber']

changed_cells = 0
for col in text_cols:
    original = df[col].astype(str)
    cleaned = original.str.strip().str.title()
    changed_cells += (original != cleaned).sum()
    df[col] = cleaned

# IDs and codes should stay uppercase (Title-casing breaks alphanumeric codes like ORD200000, SAVE10)
for col in ['OrderID', 'CustomerID', 'TrackingNumber', 'CouponCode']:
    df[col] = df[col].str.upper()
df['CouponCode'] = df['CouponCode'].replace('NO COUPON', 'No Coupon')

change_log.append({
    'Change ID': 'CR004',
    'Description': "Trimmed whitespace and applied consistent casing across all text columns",
    'Impact': f"{changed_cells} cell(s) altered for formatting consistency",
    'Status': 'Resolved'
})
print(f"Cells reformatted for whitespace/casing: {changed_cells}")

Cells reformatted for whitespace/casing: 3291


In [9]:
# --- Numeric precision (2 decimals) on monetary columns ---
df['UnitPrice'] = df['UnitPrice'].round(2)
df['TotalPrice'] = df['TotalPrice'].round(2)

change_log.append({
    'Change ID': 'CR005',
    'Description': "Rounded 'UnitPrice' and 'TotalPrice' to 2 decimal places",
    'Impact': f"{len(df)} records standardized to currency precision",
    'Status': 'Resolved'
})
df[['UnitPrice', 'TotalPrice']].head()

,UnitPrice,TotalPrice
0,570.62,2853.10
1,151.35,302.70
2,550.68,2753.40
3,273.19,273.19
4,626.01,2504.04


In [10]:
# --- Validate TotalPrice = Quantity x UnitPrice ---
expected_total = (df['Quantity'] * df['UnitPrice']).round(2)
mismatches = (expected_total != df['TotalPrice']).sum()

if mismatches > 0:
    df['TotalPrice'] = expected_total

change_log.append({
    'Change ID': 'CR006',
    'Description': "Validated TotalPrice = Quantity x UnitPrice for every record",
    'Impact': f"{mismatches} mismatch(es) found and corrected" if mismatches > 0 else "0 mismatches found — all records consistent",
    'Status': 'Resolved'
})
print(f"TotalPrice mismatches corrected: {mismatches}")

TotalPrice mismatches corrected: 0


## Phase 4 — Verification Gate
*Before you finish, you must prove there are zero duplicate IDs and zero incorrectly formatted dates.*

In [11]:
import re

dup_ids_final = df['OrderID'].duplicated().sum()
date_pattern = re.compile(r'^\d{4}-\d{2}-\d{2}$')
bad_dates_final = (~df['Date'].astype(str).str.match(date_pattern)).sum()
nulls_final = df.isnull().sum().sum()

print(f"Duplicate OrderIDs remaining : {dup_ids_final}  {'PASS' if dup_ids_final == 0 else 'FAIL'}")
print(f"Incorrectly formatted dates  : {bad_dates_final}  {'PASS' if bad_dates_final == 0 else 'FAIL'}")
print(f"Remaining null values        : {nulls_final}  {'PASS' if nulls_final == 0 else 'FAIL'}")

assert dup_ids_final == 0, "Verification gate failed: duplicate OrderIDs found"
assert bad_dates_final == 0, "Verification gate failed: malformed dates found"
assert nulls_final == 0, "Verification gate failed: null values remain"

print("\nAll verification checks passed. Dataset meets the 0% error threshold for Project 2.")

Duplicate OrderIDs remaining : 0  PASS
Incorrectly formatted dates  : 0  PASS
Remaining null values        : 0  PASS

All verification checks passed. Dataset meets the 0% error threshold for Project 2.


## Change Log
Full record of every transformation applied, for stakeholder review.

In [12]:
log_df = pd.DataFrame(change_log)
log_df

,Change ID,Description,Impact,Status
0,CR001,Imputed 309 missing 'CouponCode' values with l...,Preserved 309 records; 0 rows dropped,Resolved
1,CR002,"Audited unique identifiers (OrderID, TrackingN...","Found 0 duplicate OrderIDs, 0 duplicate Tracki...",Resolved
2,CR003,Standardized 'Date' column to ISO 8601 format ...,1200 records reformatted,Resolved
3,CR004,Trimmed whitespace and applied consistent casi...,3291 cell(s) altered for formatting consistency,Resolved
4,CR005,Rounded 'UnitPrice' and 'TotalPrice' to 2 deci...,1200 records standardized to currency precision,Resolved
5,CR006,Validated TotalPrice = Quantity x UnitPrice fo...,0 mismatches found — all records consistent,Resolved


## Export Cleaned Dataset

In [13]:
df.to_excel(OUTPUT_FILE, index=False)
print(f"Cleaned dataset saved to: {OUTPUT_FILE}")
print(f"Final shape: {df.shape}")

Cleaned dataset saved to: Dataset_for_Data_Analytics_CLEANED.xlsx
Final shape: (1200, 14)


## Summary

- Started with **1,200** rows / 14 columns.
- Imputed 309 missing `CouponCode` values (labeled `No Coupon`).
- Audited and removed duplicate rows (0 found in this dataset — verified clean).
- Standardized all dates to ISO 8601, all text fields to trimmed/consistent casing, and all monetary values to 2-decimal precision.
- Verified `TotalPrice = Quantity x UnitPrice` across every record.
- Passed the Project 2 verification gate: 0% duplicate IDs, 0% malformed dates, 0 remaining nulls.

**Deliverables:** `Dataset_for_Data_Analytics_CLEANED.xlsx` + this notebook as process documentation.